# Transformação Silver - Custos

### Descrição
Este notebook realiza a transformação de dados da camada **Bronze** para **Silver** da tabela `vcredit_custos`, focando na limpeza de caracteres inválidos (texto misturado com números), padronização decimal e garantia de unicidade.

### Objetivos
* Renomear colunas para nomes descritivos (ex: `custo` para `valor_custo`)
* Padronizar formato numérico (remover "reais" e substituir vírgula por ponto)
* Converter tipo de dados de String para Decimal
* Garantir unicidade da chave primária
* Salvar na camada Silver em formato Delta Lake

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Definição dos caminhos do Unity Catalog
catalogo = "medalhao_credit"
bronze_db = "bronze_credit"
silver_db = "silver_credit"
tabela_alvo = "vcredit_custos"

# Caminho completo da tabela na camada Bronze
path_custos = f"{catalogo}.{bronze_db}.{tabela_alvo}"

# Leitura da tabela
print(f"Lendo dados de: {path_custos}")
df_bronze = spark.read.table(path_custos)

display(df_bronze.limit(5))

### Análise Exploratória
Neste trecho iremos analisar o schema e visualizar uma amostra dos dados para identificar os padrões de "sujeira" na coluna de valor (sufixo "reais", uso de vírgula), bem como verificar a volumetria inicial.

In [0]:
print(f"Total de registros: {df_bronze.count()}")

# Schema e amostra inicial
df_bronze.printSchema()
display(df_bronze.limit(10))

# Verificando padrões na coluna 'custo' para identificar a sujeira
# Isso vai mostrar claramente os valores como '0.0026reais' misturados com '0,1816'
print("Amostra da coluna 'custo' (dados brutos):")
display(df_bronze.select("custo").sample(withReplacement=False, fraction=0.1).limit(20))

### Problemas Identificados
1.  **Formato Inconsistente:** A coluna `custo` mistura formatos. Alguns registros possuem o sufixo "reais" e ponto, outros usam vírgula como decimal.
2.  **Tipo Incorreto:** Dados numéricos estão tipados como `string`.
3.  **Nomenclatura:** O nome `custo` é genérico, vamos alterar para `valor_custo`.

---

### Transformações Aplicadas

#### 1. Limpeza e Conversão de Tipo
Nesta etapa, aplicamos uma limpeza em cascata:
1.  Removemos a string "reais".
2.  Substituímos a vírgula (`,`) por ponto (`.`).
3.  Convertemos para `Decimal(10,2)` para garantir precisão monetária.

In [0]:
# Tratamento da coluna de valor 
df_step1 = df_bronze.select(
    F.col("id_custo"),
    F.col("id_chamado"),
    
    # 1. Regex "[^0-9,.]": Seleciona tudo que nao for número, vírgula ou ponto e remove.
    #    Isso elimina 'R', '$', 'reais', espaços e qualquer caractere invisível.
    # 2. Troca ',' por '.'
    # 3. Converte para Decimal(18, 6) para suportar alta precisão (ex: 0.00000154)
    F.regexp_replace(
        F.regexp_replace(F.col("custo"), "[^0-9,.]", ""), 
        ",", "."
    ).cast(DecimalType(18, 6)).alias("valor_custo"),
    
    F.col("ingestion_timestamp")
)

# Verificando o resultado da limpeza
print("Amostra após limpeza:")
display(df_step1.limit(10))
df_step1.printSchema()

#### 2. Tratamento de Duplicatas
Garantia de que não existem IDs de custo duplicados na base.

#### 3. Ordenação
Ordenado por `id_custo` para melhor organização física dos dados.

In [0]:
# --- Passo 2: Tratamento de Duplicatas ---
# Contagem antes
total_antes = df_step1.count()

# Remover duplicatas pelo ID (Chave Primária)
# Criamos o df_silver final a partir daqui
df_silver = df_step1.dropDuplicates(["id_custo"])

# Contagem depois
total_depois = df_silver.count()

print(f"Registros antes: {total_antes}")
print(f"Registros depois: {total_depois}")
print(f"Duplicatas removidas: {total_antes - total_depois}")

# --- Passo 3: Ordenação ---
df_silver = df_silver.orderBy(F.col("id_custo"))

display(df_silver.limit(10))

### Análise Completa de Qualidade - Custos
Agora que os dados estão limpos, calculamos métricas de qualidade e estatísticas financeiras para garantir que não perdemos dados importantes e que os valores fazem sentido.

In [0]:
# Análise completa de qualidade e estatísticas financeiras
qualidade_dados = df_silver.select([
    F.count("*").alias("total_registros"),
    F.countDistinct("id_custo").alias("id_custo_unicos"),
    F.countDistinct("id_chamado").alias("id_chamado_unicos"),
    F.avg("valor_custo").alias("custo_medio"),
    F.sum("valor_custo").alias("custo_total"),
    F.min("valor_custo").alias("custo_minimo"),
    F.max("valor_custo").alias("custo_maximo")
]).collect()[0]

print("RELATÓRIO DE QUALIDADE:")
print(f"Total de registros: {qualidade_dados['total_registros']}")
print(f"IDs custo únicos: {qualidade_dados['id_custo_unicos']}")
print(f"IDs chamado únicos: {qualidade_dados['id_chamado_unicos']}")
print("-" * 30)
print("ESTATÍSTICAS FINANCEIRAS:")
print(f"Custo Médio: R$ {qualidade_dados['custo_medio']:.2f}")
print(f"Custo Mínimo: R$ {qualidade_dados['custo_minimo']:.2f}")
print(f"Custo Máximo: R$ {qualidade_dados['custo_maximo']:.2f}")
print(f"Investimento Total Monitorado: R$ {qualidade_dados['custo_total']:.2f}")

# Verificar integridade 
nulos = df_silver.filter(F.col("id_chamado").isNull()).count()
print("-" * 30)
print(f"Registros órfãos (sem id_chamado): {nulos}")

In [0]:
# Salvando na silver
tabela_destino = f"{catalogo}.{silver_db}.vcredit_custos"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_destino)

print(f"✅ Tabela {tabela_destino} salva com sucesso!")